# KaVa Phase 2 — Colab + Google Drive persistence

This notebook is fixed to **KaVa** so it cannot restore or overwrite the other method. The training cell remains active on Colab's server, so you may close your browser after training starts. Colab can still terminate managed runtimes; every completed atomic checkpoint is therefore mirrored to Drive and the next runtime resumes from it.

Before the first run, upload the complete extracted checkpoint folder to:

`MyDrive/CODI_KAVA/uploads/step_00024000.pt/`

Upload the entire folder, including `data.pkl`, `data/`, `byteorder`, `version`, and hidden metadata files. A folder name without `.pt` is also accepted. The runner rebuilds and PyTorch-verifies a real `.pt` archive before persistence.

In [ ]:
# Fixed KaVa experiment settings. Do not change METHOD while resuming.
METHOD = "kava"
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"            # After pushing this notebook, replace with that exact commit SHA.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"
LOCAL_ROOT = "/content/codikava_runtime"
MAX_SECONDS = 32400             # 9h, with the trainer's 5% safety margin.
EVAL_LIMIT = 200                # 200 quick gate; 0 full eval; -1 skip eval.
assert METHOD == "kava"


## 1. Mount Drive and prepare the repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib, subprocess, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("For later sessions, set RUN_COMMIT to:", commit)


## 2. GPU and checkpoint preflight

In [ ]:
import json, torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

expected_step = 24000
uploads = pathlib.Path(DRIVE_ROOT) / "uploads"
upload_candidates = [uploads / f"step_{expected_step:08d}.pt", uploads / f"step_{expected_step:08d}", uploads / f"step_{expected_step:08d}.zip"]
durable = pathlib.Path(DRIVE_ROOT) / "outputs" / METHOD / "checkpoints" / f"step_{expected_step:08d}.pt"
upload = next((path for path in upload_candidates if path.exists()), None)
if not durable.is_file() and upload is None:
    raise FileNotFoundError(f"Upload the extracted step_{expected_step:08d}.pt folder to {uploads}")
print("Resume source:", durable if durable.is_file() else upload)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


## 3. Start or resume training

This cell intentionally stays busy. Wait until the log shows `[resume] continuing from step ...` or the first loss message, then you may close the browser. A detached/nohup process is deliberately not used because an inactive cell makes Colab more likely to consider the runtime idle.

Progress is durable under `MyDrive/CODI_KAVA/outputs/<method>/`; logs and status are under `logs/` and `status/`. Exit code `42` means the 9-hour guard saved successfully—open a new runtime later and rerun cells 1–3.

In [ ]:
import datetime, subprocess, time
logs = pathlib.Path(DRIVE_ROOT) / "logs"
logs.mkdir(parents=True, exist_ok=True)
log_path = logs / f"{METHOD}.log"
cmd = [
    sys.executable, "-u", "scripts/colab_runner.py",
    "--method", METHOD,
    "--drive-root", DRIVE_ROOT,
    "--local-root", LOCAL_ROOT,
    "--max-seconds", str(MAX_SECONDS),
    "--eval-limit", str(EVAL_LIMIT),
    "--allow-environment-change",
]
print("Starting:", " ".join(cmd), flush=True)
print("Persistent log:", log_path, flush=True)
with log_path.open("a", encoding="utf-8", buffering=1) as log:
    log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(cmd)} ===\n")
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    last_flush = time.monotonic()
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        if time.monotonic() - last_flush >= 30:
            log.flush()
            last_flush = time.monotonic()
    return_code = process.wait()
    log.flush()
print("Session exit code:", return_code)
print("0 = training complete (and evaluation completed); 42 = durable checkpoint saved, rerun later")


## 4. Inspect durable state after the session

In [ ]:
status_path = pathlib.Path(DRIVE_ROOT) / "status" / f"{METHOD}.json"
print(json.dumps(json.loads(status_path.read_text()), indent=2) if status_path.is_file() else "No status file")
drive_output = pathlib.Path(DRIVE_ROOT) / "outputs" / METHOD
for path in sorted(drive_output.rglob("*")):
    if path.is_file() and not path.name.endswith((".uploading", ".tmp")):
        print(f"{path.relative_to(drive_output)}  {path.stat().st_size / 2**20:.1f} MiB")


## KaVa completion sequence

1. Rerun this notebook after each exit code `42` until Drive status is `complete` and the durable checkpoint is step 96,405.
2. The default `EVAL_LIMIT = 200` creates the comparable quick evaluation only after KaVa completes.
3. After the quick gate is safe, rerun the completed checkpoint with `EVAL_LIMIT = 0` for full evaluation.
4. CODI belongs in `colab_phase2_codi.ipynb`; never change this notebook's method.

Closing the browser does not guarantee a managed Colab VM will survive. Drive mirroring protects the latest completed checkpoint, but a terminated runtime must still be restarted manually.